## Module 3 Assignment - Building RAG systems with a Vector Database

---

This notebook follows Module 3 of DeepLearning.AI's Retrieval-Augmented Generation course, which builds a small RAG system on top of a vector database. In the original course this meant working with a Weaviate collection; this local adaptation uses a local Chroma collection instead, built from the same BBC News data (already included under `data/` in this repository). The notebook:

- Loads a collection of BBC News data, already chunked.
- Retrieves documents from the vector database.
- Implements Semantic Search, BM25, and Hybrid Search (using RRF) against it.
- Uses a local LLM to generate responses.

The original course pointed to two Ungraded Labs (on the Weaviate API, and on chunking) for background on these topics; those labs live on DeepLearning.AI's platform and aren't part of this repository.


# Table of Contents
- [ 1 - Loading the libraries](#1)
- [ 2 - Setting up the local vector store and loading the data](#2)
  - [ 2.1 Loading the local vector store](#2-1)
  - [ 2.2 Loading the data](#2-2)
- [ 3 - Loading the Collection](#3)
  - [ 3.1 Metadata filtering](#3-1)
    - [ Exercise 1](#ex01)
  - [ 3.2 Semantic search](#3-2)
    - [ Exercise 2](#ex02)
  - [ 3.3 BM25 Serach](#3-3)
    - [ Exercise 3](#ex03)
  - [ 3.4 Hybrid search](#3-4)
    - [ Exercise 4](#ex04)
    - [ Exercise 5](#ex05)
- [ 4 - Incorporating the local vector store into our previous schema](#4)
  - [ 4.1 Generating the final prompt](#4-1)
  - [ 4.2 LLM call](#4-2)
- [ 5 - Experimenting with Your RAG System](#5)



---
<h4 style="color:black; font-weight:bold;">USING THE TABLE OF CONTENTS</h4>

JupyterLab still provides an easy way to navigate this notebook: the Table of Contents tab, in the left panel.

---

<h4 style="color:green; font-weight:bold;">ABOUT GRADING IN THE ORIGINAL COURSE</h4>

In the original DeepLearning.AI course, this point in the notebook carried tips for the automated grader — global variables had to stay in UPPERCASE, extra cells were ignored by the grader, and submitting meant saving the notebook and clicking a "Submit assignment" button. This local adaptation has no grader and nothing to submit, so none of that applies here; the notebook is meant to be read and run end to end.
---

<a id='1'></a>
## 1 - Loading the libraries

---

Run the cell below to load the necessary libraries for this assignment.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    import joblib

import numpy as np
import pandas as pd

from chroma_store import ChromaStore
from embedding import embed_query
from retrieval import (
    bm25_retrieve,
    clear_bm25_cache,
    filter_by_metadata,
    generate_final_prompt,
    hybrid_retrieve,
    llm_call,
    semantic_search_retrieve,
    semantic_search_with_reranking,
)
from utils import (
    generate_with_single_input,
    print_object_properties,
)

# Use the central config for path resolution to avoid working-directory issues
from setting import config



resource module not available on Windows


<a id='2'></a>
## 2 - Setting up the local vector store and loading the data

---

In this section, you will set up the local vector store and load the data, which consists of the [BBC news dataset](https://www.kaggle.com/datasets/gpreda/bbc-news) adapted from Kaggle, already included under `data/` in this repository.


<a id='2-1'></a>
### 2.1 Loading the local vector store

The original course connected to a Weaviate client here, with a server already running in the background. This local adaptation instead opens a local Chroma store — no server or client connection needed.

**Troubleshooting:**

- If you encounter issues loading the next cell, try restarting your kernel, clicking in the circled arrow in the panel above.


In [2]:

store = ChromaStore(persist_directory=str(config.chromaPath))
    
# client = weaviate.connect_to_local(port=8079, grpc_port=50050)

<a id='2-2'></a>
### 2.2 Loading the data

Now, let's load the data. The dataset is structured with the following fields:

- **`title`**: The headline of the article.
- **`pubDate`**: The publication date and time of the article.
- **`guid`**: A unique identifier for the article, commonly used for listing.
- **`link`**: A URL link to access the full article online.
- **`description`**: A brief summary or teaser of the article's content.
- **`article_content`**: The complete text of the article, providing detailed information.

In [ ]:

from embedding import get_embedding


# 2. Load the CSV as before
df = pd.read_csv(str(config.newsCSV))

# 3. Extract the text list – replace 'text_column' with the actual column name
texts = df['description'].tolist()   # e.g., df['content'] or df['article']

# 4. Compute embeddings (cached automatically)
embeddings = get_embedding(texts)   # returns np.ndarray of shape (len(texts), 384)

# 5. Build the list of dicts (bbc_data) – you can still use the DataFrame rows
bbc_data = [row.to_dict() for _, row in df.iterrows()]



In [5]:
print(f"First item: {bbc_data[0]}")

# from utils import print_object_properties
# print_object_properties(bbc_data[0])

First item: {'guid': 'e3dc5caa18f9a16d7edcc09f8d5c2bb4', 'title': "Harvey Weinstein's 2020 rape conviction overturned", 'description': 'Victims group describes the New York appeal court\'s decision to retry Hollywood mogul as "profoundly unjust".', 'venue': 'BBC', 'url': 'https://www.bbc.co.uk/news/world-us-canada-68899382', 'published_at': '2024-04-25 18:24:04+00', 'updated_at': '2024-04-26 20:03:00.628113+00'}


<a id='3'></a>
## 3 - Loading the Collection

---

In this section, you will load the collection containing the BBC News dataset.

In [6]:
# Transform data and populate Chroma collection
collection_name = "bbc_collection"
store.create_collection(collection_name)

# Transform rows into assignment document shape
documents = []
for row in bbc_data:
    doc = {
        "id": str(row.get("guid", "")),
        "title": row.get("title", ""),
        "chunk": row.get("description", row.get("chunk", "")),
        "pubDate": str(row.get("published_at", "")),
        "link": row.get("url", row.get("link", "")),
    }
    documents.append(doc)

store.add_documents(collection_name, documents, embeddings.astype(np.float32))
collection = collection_name  # alias for compatibility with usage cells


# collection = client.collections.get("bbc_collection")

In [7]:

print(f"The number of elements in the collection is: {store.count(collection)}")

# print(f"The number of elements in the collection is: {len(collection)}")

The number of elements in the collection is: 870


Let's fetch one example of object in this collection.

In [8]:
from utils import print_object_properties
docs = store.get_all_documents(collection)
print("Printing the properties (some will be truncated due to size)")
if docs:
    print_object_properties(docs[0])
else:
    print("Collection is empty.")


        # object = collection.query.fetch_objects(limit=1, include_vector=True).objects[0]
        # print("Printing the properties (some will be truncated due to size)")
        # print_object_properties(object.properties)
        # print("Vector: (truncated)", object.vector["main_vector"][0:15])
        # print("Vector length: ", len(object.vector["main_vector"]))

Printing the properties (some will be truncated due to size)
chunk: Victims group describes the New York appeal court's decision to retry Hollywood mogul as "profoundly...(truncated)
id: e3dc5caa18f9a16d7edcc09f8d5c2bb4
link: https://www.bbc.co.uk/news/world-us-canada-68899382
pubDate: 2024-04-25 18:24:04+00
title: Harvey Weinstein's 2020 rape conviction overturned



The vector length is `384` in this local adaptation (the original course's Weaviate-based version used a 768-dimension vector). So every chunk in the vector database is mapped into a 384-dimension vector using `sentence-transformers/all-MiniLM-L6-v2`. This is the vector this local Chroma store uses to perform semantic search.

<a id='ex01'></a>

<a id='3-1'></a>
### 3.1 Metadata filtering

<a id='ex01'></a>
### Exercise 1

In this exercise, you will implement a metadata filtering function. This function will take several inputs: a property (such as `article_content`, `title`, `pubDate`, etc.), the values you want to filter by, the collection you want to search in, and the number of items you want to retrieve.

<details>
<summary style="color: green;">Hint 1</summary>
Remember that to perform filtering based only on metadata, the appropriate method to use is <code>collection.query.fetch_objects</code>.
</details>
<details>
<summary style="color: green;">Hint 2</summary>
When using <code>collection.query.fetch_objects</code>, you must provide the <code>metadata_property</code> as the <code>property</code> and the corresponding <code>Filter</code> object.
</details>
<details>
<summary style="color: green;">Hint 3</summary>
The filter object should be used with the method <code>.by_property</code> for the appropriate property, and <code>.contains_any</code> with the relevant values. A typical call would be <code>Filter.by_property(metadata_property).contains_any(values)</code>.
To limit the results, use <code>limit=limit</code> within the <code>.fetch_objects</code> method.
</details>

*(This local adaptation's solved cell instead filters through `retrieval.filter_by_metadata`, backed by the local Chroma store.)*

In [9]:
"""Code cell 19 from the notebook."""
# GRADED CELL

def filter_by_metadata(
    metadata_property: str,
    values: list[str],
    collection: "weaviate.collections.collection.sync.Collection",
    limit: int = 5,
) -> list:
    """
    Retrieves objects from a specified collection based on metadata filtering criteria.
    """
    from retrieval import filter_by_metadata as _adapted_filter

    return _adapted_filter(
        metadata_property=metadata_property,
        values=values,
        store=store,
        collection_name=collection,
        limit=limit,
    )

# Ensure the function is available in the outer scope for later cells
if "filter_by_metadata" not in dir():
    pass  # defined by the def above


In [10]:
from utils import print_object_properties
# Let's get an example
res = filter_by_metadata("title", ["Taylor Swift"], collection, limit=2)
for x in res:
    print_object_properties(x)

**Expected output**
```
article_content: The 2024 awards season kicked off in style at the Golden Globes - the first major red carpet event o...(truncated)
chunk: some of his previous get-ups. The Bear's Jeremy Allen White - who recently became the new face (and ...(truncated)
chunk_index: 4
description: Stars including Margot Robbie and Taylor Swift arrived in a variety of eye-catching outfits.
link: https://www.bbc.co.uk/news/entertainment-arts-67908727?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-01-08 03:23:58+00:00
title: Margot Robbie, Taylor Swift and more on Golden Globes red carpet

article_content: The 2024 awards season kicked off in style at the Golden Globes - the first major red carpet event o...(truncated)
chunk: headpiece - not entirely a fashion choice. She says the "protective veil" is because she hurt her fa...(truncated)
chunk_index: 5
description: Stars including Margot Robbie and Taylor Swift arrived in a variety of eye-catching outfits.
link: https://www.bbc.co.uk/news/entertainment-arts-67908727?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-01-08 03:23:58+00:00
title: Margot Robbie, Taylor Swift and more on Golden Globes red carpet
```

<a id='ex02'></a>

<a id='3-2'></a>
### 3.2 Semantic search

<a id='ex02'></a>
### Exercise 2

In this exercise, you will implement a semantic search retrieval, similar to the one you created in the previous assignment, but this time utilizing the Weaviate API.

<details>
<summary style="color: green;">Hint</summary>
Remember that to perform semantic search, you should use the method <code>collection.query.near_text</code>.
The <code>top_k</code> parameter in the function dictates how many results to retrieve. In Weaviate, this is referred to as <code>limit</code>. Adjust this parameter as needed.
</details>

*(This local adaptation's solved cell instead calls `retrieval.semantic_search_retrieve`, backed by the local Chroma store.)*

In [12]:
"""Code cell 24 from the notebook."""
# GRADED CELL

def semantic_search_retrieve(
    query: str,
    store: "ChromaStore",
    collection_name: str,
    embed_function,
    top_k: int = 5,
) -> list:
    """
    Performs a semantic search on a collection and retrieves the top relevant chunks.
    """
    from retrieval import semantic_search_retrieve as _adapted_ss

    return _adapted_ss(
        query=query,
        store=store,
        collection_name=collection_name,
        embed_function=embed_function,
        top_k=top_k,
    )



In [13]:
"""Code cell 25 from the notebook."""
# Let's have an example!
print_object_properties(
    semantic_search_retrieve(
        query="Tell me about the last Taylor Swift show",
        collection_name=collection,
        top_k=2,
        store=store,
        embed_function= embed_query
    )
)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

chunk: As the boss of PC Music, the godfather of hyperpop confounded critics but won over Beyoncé and Charl...(truncated)
id: 69ef9e3d80afb076905b3875a1cbe1e7
link: https://www.theguardian.com/music/2024/apr/26/ag-cook-pc-music-britpop-interview
pubDate: 2024-04-26 10:00:36+00
title: ‘People think I hate pop’: super-producer AG Cook on working with Beyoncé and honouring his friend Sophie

chunk: Surprise reversal of producer’s New York conviction led to anger from stars and accusers, including ...(truncated)
id: 23bad9907f46cac7c4512e7dd1edb687
link: https://www.theguardian.com/film/2024/apr/25/harvey-weinstein-rape-conviction
pubDate: 2024-04-25 19:37:39+00
title: Hollywood reacts to overturning of Harvey Weinstein rape conviction: ‘Beyond disappointed’




**Expected Output**
```
article_content: Taylor Swift has finished the European leg of her Eras Tour with a record-breaking show at Wembley S...(truncated)
chunk: size crowd at all". At an earlier show in Liverpool, she had also called the Eras Tour the “most exh...(truncated)
chunk_index: 10
description: The star is joined by Florence + The Machine and sings So Long, London at her final UK show.
link: https://www.bbc.com/news/articles/cr5nr3n6epvo
pubDate: 2024-08-21 03:02:08+00:00
title: 'I've never had it this good' - Taylor Swift thanks fans after new Wembley record

article_content: Taylor Swift has finished the European leg of her Eras Tour with a record-breaking show at Wembley S...(truncated)
chunk: regular part of the setlist. Last week, the star was joined by Ed Sheeran to play the songs Endgame ...(truncated)
chunk_index: 4
description: The star is joined by Florence + The Machine and sings So Long, London at her final UK show.
link: https://www.bbc.com/news/articles/cr5nr3n6epvo
pubDate: 2024-08-21 03:02:08+00:00
title: 'I've never had it this good' - Taylor Swift thanks fans after new Wembley record

```

<a id='ex03'></a>

<a id='3-3'></a>
### 3.3 BM25 Serach

<a id='ex03'></a>
### Exercise 3

In this exercise, you will implement a BM25 retrieval, similar to the one you created in the previous assignment, but now using the Weaviate API.
<details>
<summary style="color: green;">Hint</summary>
To perform a BM25 search, use the method <code>collection.query.bm25</code>.
The <code>top_k</code> parameter in the function specifies how many results to retrieve. In Weaviate, this parameter is referred to as <code>limit</code>. Adjust this accordingly.
</details>

*(This local adaptation's solved cell instead calls `retrieval.bm25_retrieve`, backed by the local Chroma store.)*

In [15]:
"""Code cell 29 from the notebook."""
# GRADED CELL

def bm25_retrieve(
    query: str,
    store: "ChromaStore",
    collection_name: str,
    top_k: int = 5,
) -> list:
    """
    Performs a BM25 search on a collection and retrieves the top relevant chunks.
    """
    from retrieval import bm25_retrieve as _adapted_bm25

    clear_bm25_cache()
    return _adapted_bm25(
        query=query,
        store=store,
        collection_name=collection_name,
        top_k=top_k,
    )



In [16]:
"""Code cell 30 from the notebook."""
print_object_properties(
    bm25_retrieve(
        "Tell me about the last Taylor Swift show",
        collection_name=collection,
        top_k=2,
        store=store,
    )
)

Split strings:   0%|          | 0/870 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/870 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/870 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

chunk: As Taylor Swift tops $1bn in tour revenue, musicians playing smaller venues are facing pitiful fees ...(truncated)
id: 927257674585bb6ef669cf2c2f409fa7
link: https://www.theguardian.com/music/2024/apr/25/shocking-truth-money-bands-make-on-tour-taylor-swift
pubDate: 2024-04-25 09:39:04+00
title: ‘The working class can’t afford it’: the shocking truth about the money bands make on tour

chunk: From Miriam being ‘revealed’ as a transgender woman to the contestants trashing the set, it was the ...(truncated)
id: 567d5389d63e2304a909086e59a48644
link: https://www.theguardian.com/tv-and-radio/2024/apr/25/she-was-tough-but-it-broke-her-why-theres-something-about-miriam-was-reality-tvs-most-shameful-low
pubDate: 2024-04-25 14:52:37+00
title: ‘She was tough, but it broke her’: why There’s Something About Miriam was reality TV’s most shameful low




**Expected Output**
```
article_content: Rapper Killer Mike won three Grammys in the rap category - best rap song, best rap performance and b...(truncated)
chunk: police brutality and systemic racism. He was a highly visible supporter of Bernie Sanders' two campa...(truncated)
chunk_index: 4
description: The 48-year-old was detained on a misdemeanour charge after winning three awards in the rap category.
link: https://www.bbc.co.uk/news/world-us-canada-68201021?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-02-05 23:27:08+00:00
title: Killer Mike dismisses arrest at Grammys as 'speed bump'

article_content: Rapper Killer Mike won three Grammys in the rap category - best rap song, best rap performance and b...(truncated)
chunk: Nicki Minaj. He also won a third award for best rap album with his album Michael. "You cannot tell m...(truncated)
chunk_index: 3
description: The 48-year-old was detained on a misdemeanour charge after winning three awards in the rap category.
link: https://www.bbc.co.uk/news/world-us-canada-68201021?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-02-05 23:27:08+00:00
title: Killer Mike dismisses arrest at Grammys as 'speed bump'
```

<a id='ex04'></a>

<a id='3-4'></a>
### 3.4 Hybrid search

<a id='ex04'></a>
### Exercise 4

In this exercise, you will implement a Reciprocal Rank Fusion (RRF) retrieval system using the Weaviate API. To achieve this, you will need to use the `collection.query.hybrid` method.

<details>
<summary style="color: green;">Hint</summary>
To perform a hybrid search, use the method <code>collection.query.hybrid</code>.
The <code>top_k</code> parameter in the function specifies how many results to retrieve. In Weaviate, this is referred to as <code>limit</code>. Make sure to also include the <code>alpha</code>.
</details>

*(This local adaptation's solved cell instead calls `retrieval.hybrid_retrieve`, backed by the local Chroma store.)*

In [18]:
"""Code cell 34 from the notebook."""
# GRADED CELL

def hybrid_retrieve(
    query: str,
    store: "ChromaStore",
    collection_name: str,
    embed_function,
    alpha: float = 0.5,
    top_k: int = 5,
) -> list:
    """
    Performs a hybrid search on a collection and retrieves the top relevant chunks.
    """
    from retrieval import hybrid_retrieve as _adapted_hybrid

    return _adapted_hybrid(
        query=query,
        store=store,
        collection_name=collection_name,
        embed_function=embed_function,
        alpha=alpha,
        top_k=top_k,
    )



In [19]:
"""Code cell 35 from the notebook."""
print_object_properties(
    hybrid_retrieve(
        "Tell me about the last Taylor Swift show",
        top_k=2,
        collection_name=collection,
        embed_function= embed_query,
        store=store
    )
)



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

chunk: As the boss of PC Music, the godfather of hyperpop confounded critics but won over Beyoncé and Charl...(truncated)
id: 69ef9e3d80afb076905b3875a1cbe1e7
link: https://www.theguardian.com/music/2024/apr/26/ag-cook-pc-music-britpop-interview
pubDate: 2024-04-26 10:00:36+00
title: ‘People think I hate pop’: super-producer AG Cook on working with Beyoncé and honouring his friend Sophie

chunk: From Miriam being ‘revealed’ as a transgender woman to the contestants trashing the set, it was the ...(truncated)
id: 567d5389d63e2304a909086e59a48644
link: https://www.theguardian.com/tv-and-radio/2024/apr/25/she-was-tough-but-it-broke-her-why-theres-something-about-miriam-was-reality-tvs-most-shameful-low
pubDate: 2024-04-25 14:52:37+00
title: ‘She was tough, but it broke her’: why There’s Something About Miriam was reality TV’s most shameful low




**Expected Output**
```
article_content: Rapper Killer Mike won three Grammys in the rap category - best rap song, best rap performance and b...(truncated)
chunk: police brutality and systemic racism. He was a highly visible supporter of Bernie Sanders' two campa...(truncated)
chunk_index: 4
description: The 48-year-old was detained on a misdemeanour charge after winning three awards in the rap category.
link: https://www.bbc.co.uk/news/world-us-canada-68201021?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-02-05 23:27:08+00:00
title: Killer Mike dismisses arrest at Grammys as 'speed bump'

article_content: Taylor Swift has finished the European leg of her Eras Tour with a record-breaking show at Wembley S...(truncated)
chunk: size crowd at all". At an earlier show in Liverpool, she had also called the Eras Tour the “most exh...(truncated)
chunk_index: 10
description: The star is joined by Florence + The Machine and sings So Long, London at her final UK show.
link: https://www.bbc.com/news/articles/cr5nr3n6epvo
pubDate: 2024-08-21 03:02:08+00:00
title: 'I've never had it this good' - Taylor Swift thanks fans after new Wembley record
```

### 5 - Reranking

<a id='ex05'></a>

<a id='ex05'></a>
### Exercise 5

In this section, you will create a new version of `semantic_search` that allows reranking of the results. This new function must support using a different query for reranking or reranking based on a specific document property (e.g., reranking using only the title).

Your task is to add the `rerank` parameter to the `collection.query.near_text` call.

<details>
<summary style="color: green;">Hint 1</summary>
<p>Remember that <code>collection.query.near_text</code> takes a query, a limit (i.e., <code>top_k</code>), and now also requires the <code>rerank</code> parameter.</p>
</details>

<details>
<summary style="color: green;">Hint 2</summary>
<p>The <code>Rerank</code> object is already loaded into memory. It takes two parameters: the query and the document property to use for ranking—<code>query</code> and <code>prop</code>, respectively.</p>
</details>

<details>
<summary style="color: green;">Hint 3</summary>
<p>Define the reranker as <code>reranker = Reranker(appropriate_parameters)</code>. Don’t forget: the query for the reranker should be <code>rerank_query</code>!</p>
</details>

*(This local adaptation's solved cell instead calls `retrieval.semantic_search_with_reranking`, backed by the local Chroma store.)*

In [21]:
"""Code cell 39 from the notebook."""
# GRADED CELL

def semantic_search_with_reranking(
    query: str,
    rerank_property: str,
    collection: "weaviate.collections.collection.sync.Collection",
    rerank_query: str = None,
    top_k: int = 5,
) -> list:
    """
    Performs a semantic search and reranks the results based on a specified property.
    """
    from retrieval import semantic_search_with_reranking as _adapted_rr

    return _adapted_rr(
        query=query,
        rerank_property=rerank_property,
        store=store,
        collection_name=collection,
        embed_function=embed_query,
        rerank_query=rerank_query,
        top_k=top_k,
    )



The reranker model receives a query and a passage (in our case, a chunk of the result) to compute a similarity score.

In [22]:
"""Code cell 41 from the notebook."""
# Set a query
query = "Tell me about the conflicts in Latin America"
# Get the results from a search (in this case the hybrid search)
results = semantic_search_with_reranking(
    query,
    collection=collection,
    top_k=2,
    rerank_property="chunk",
)



In [23]:
"""Code cell 42 from the notebook."""
print_object_properties(results)



chunk: Gustavo Gorriti, the country’s most prominent journalist, is under investigation into claims he trad...(truncated)
id: 92d273992e5920975d2bff68d3adedc1
link: https://www.washingtonpost.com/world/2024/04/24/gustavo-gorriti-peru-journalist-press-freedom/
pubDate: 2024-04-24 10:00:32+00
title: In criminal probe of reporter, advocates see attack on Peru’s democracy

chunk: The leader of the Socialist Party announced he is rethinking his position after a judge opened a cas...(truncated)
id: bb1d135856e59637b66b57dd732618df
link: https://english.elpais.com/spain/2024-04-24/spanish-prime-minister-pedro-sanchez-considers-resigning-due-to-the-unprecedented-attacks-against-his-wife-by-the-right-and-the-far-right.html
pubDate: 2024-04-24 20:44:09+00
title: Spanish PM Pedro Sánchez considers resigning due to ‘the unprecedented attacks’ against his wife




**Expected Results**
```
article_content: A huge diplomatic row has erupted after Spain's transport minister suggested Argentina's president h...(truncated)
chunk: weeks' to attend the launch of Vox's European election campaign, newspaper El Pais reported. Mr Mile...(truncated)
chunk_index: 3
description: A row breaks out after Spain's transport minister suggests Argentina's president has taken drugs.
link: https://www.bbc.com/news/articles/czd8qzvpl4lo
pubDate: 2024-05-04 15:56:45+00:00
title: Spain-Argentina row over drug-use accusation

article_content: Opposition supporters have gathered across Venezuela to protest against Nicolás Maduro's disputed vi...(truncated)
chunk: the world, from Australia to Spain and also in the United Kingdom, Canada, Colombia, Mexico and Arge...(truncated)
chunk_index: 4
description: Opposition leader María Corina Machado joined thousands of demonstrators in the capital Caracas.
link: https://www.bbc.com/news/articles/cgedgqqy7x9o
pubDate: 2024-08-17 23:19:48+00:00
title: Protests across Venezuela as election dispute goes on
```

<a id='4'></a>
## 4 - Incorporating the local vector store into our previous schema
---

This section is not graded. Here, you will revisit the functions used throughout the assignments to integrate the local vector store into your existing schema. Once integrated, you will be able to run prompts and test your new RAG system!

<a id='4-1'></a>
### 4.1 Generating the final prompt



In [25]:
"""Code cell 46 from the notebook."""

def generate_final_prompt(
    query: str,
    top_k: int,
    retrieve_function: callable,
    rerank_query: str = None,
    rerank_property: str = None,
    use_rerank: bool = False,
    use_rag: bool = True,
) -> str:
    """
    Generates a final prompt by optionally retrieving and formatting relevant documents
    using retrieval-augmented generation (RAG).
    """
    from retrieval import generate_final_prompt as _adapted_gfp

    return _adapted_gfp(
        query=query,
        top_k=top_k,
        retrieve_function=retrieve_function,
        store=store,
        collection_name=collection,
        embed_function=embed_query,
        rerank_query=rerank_query,
        rerank_property=rerank_property,
        use_rerank=use_rerank,
        use_rag=use_rag,
    )



In [26]:
"""Code cell 47 from the notebook."""
prompt = generate_final_prompt(
    "Tell me the economic situation of the US in 2024.",
    top_k=5,
    retrieve_function=semantic_search_retrieve,
    use_rerank=False,
    rerank_property="title",
)



In [27]:
"""Code cell 48 from the notebook."""
print(prompt)



Answer the user query below. There will be provided additional information for you to compose your answer. The relevant information provided is from 2024 and it should be added as your overall knowledge to answer the query, you should not rely only on this information to answer the query, but add it to your overall knowledge.The news data is ordered by relevance.Query: Tell me the economic situation of the US in 2024.
2024 News: Title: America's Economy Is No. 1. That Means Trouble, Chunk: If you want a single number to capture America’s economic stature, here it is: This year, the U.S. will account for 26.3% of the global gross domestic product, the highest in almost two decades. That’s based on the latest projections from the International Monetary Fund. According to the IMF, Europe’s share of world GDP has dropped 1.4 percentage points since 2018, and Japan’s by 2.1 points. The U.S. share, by contrast, is up 2.3 points., Published at: 2024-04-26 01:04:00+00
URL: https://www.wsj.com/

<a id='4-2'></a>
### 4.2 LLM call

Let's revisit the `llm_call` function, now adapted to this assignment.

In [28]:
"""Code cell 50 from the notebook."""
from utils import ollama_llm_backend

def llm_call(
    query: str,
    retrieve_function: callable = None,
    top_k: int = 5,
    use_rag: bool = True,
    use_rerank: bool = False,
    rerank_property: str = None,
    rerank_query: str = None,
) -> str:
    """
    Simulates a call to a language model by generating a prompt and using it to produce a response.
    """
    from retrieval import llm_call as _adapted_llm

    return _adapted_llm(
        query=query,
        retrieve_function=retrieve_function,
        store=store,
        collection_name=collection if isinstance(collection, str) else None,
        embed_function=embed_query,
        top_k=top_k,
        use_rag=use_rag,
        use_rerank=use_rerank,
        rerank_property=rerank_property,
        rerank_query=rerank_query,
        llm_backend=ollama_llm_backend,  # returns prompt; real LLM call would need config
    )



In [29]:
"""Code cell 51 from the notebook."""
query = "Tell me about United States and Brazil's relationship over the course of 2024. Provide links for the resources you use in the answer."



In [30]:
"""Code cell 52 from the notebook."""
# Result with reranked results
print(
    llm_call(
        query=query,
        top_k=5,
        retrieve_function=hybrid_retrieve,
    )
)



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Based on the information provided in the news, there is no direct link between the United States and Brazil's relationship over the course of 2024. The events described only pertain to different geopolitical situations involving Ukraine, migration from Africa, China-US relations, and US clean energy projects. There are no specific reports or data available that suggest a significant interaction or relationship development between these two nations during 2024.

Therefore, the information provided does not offer any insight into how United States and Brazil's relationship was evolving in 2024. For more comprehensive understanding of this period, one would need to look at specific news articles, diplomatic statements, official reports, or other reliable sources that cover events relating to these two countries' relationships during 2024.


<a id='5'></a>
## 5 - Experimenting with Your RAG System

Now it is time for you to experiment with the system! Run the next cell to load a widget that will input a query, a rerank property, and output five different LLM responses:

1. With semantic search
2. With semantic search and reranking
3. With BM25 search
4. With hybrid search
5. Without RAG


In [ ]:
# """Code cell 54 from the notebook."""
# display_widget(
#     llm_call,
#     semantic_search_retrieve,
#     bm25_retrieve,
#     hybrid_retrieve,
#     semantic_search_with_reranking,
# )



This completes Module 3 of the original course: the RAG pipeline now retrieves through a vector store, supporting metadata filtering, semantic search, BM25, hybrid search, and reranking.